Fine-tunning de um SLM

In [1]:
import re
import torch; 
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu132
!pip install unsloth

/bin/bash: /home/wytcor/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home/wytcor/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home/wytcor/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home/wytcor/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home/wytcor/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
ERROR: Invalid requirement: 'trl==0.22.': Expected comma (within version specifier), semicolon (after version specifier) or end
    trl==0.22.
       ~~~~~~^
/bin/bash: /home/wytcor/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Looking in indexes: https://download.pytorch.org/whl/cu132
/bin/bash: /home/wytcor/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [2]:
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

/bin/bash: /home/wytcor/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 MB 22.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 1.2 MB/s  0:00:01 eta 0:00:01m
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2026.9.2
    Uninstalling unsloth_zoo-2026.9.2:
      Successfully uninstalled unsloth_zoo-2026.9.2
  Attempting uninstall: unsloth━━━━━━━━━━━━━━━━━ 0/2 [unsloth_zoo]
    Found existing installation: unsloth 2026.9.3━━━━━━━━━━━━━━━━━ 1/2 [unsloth]
    Uninstalling unsloth-2026.9.3:━━━━━━━━━━━━━━━━━━━ 1/2 [unsloth]
      Successfully uninstalled unsloth-2026.9.3━━━━━━━━━━━━━━━ 1/2 [unsloth]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [unsloth]m1/2 [unsloth]


In [3]:
!pip install unsloth

/bin/bash: /home/wytcor/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [3]:
import torch

torch.cuda.is_available()

True

In [4]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-0.6B-unsloth-bnb-4bit"
]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B",
    max_seq_length = 1024,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/wytcor/miniconda3/envs/anon-lib/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.3: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3050 Laptop GPU. Num GPUs = 1. Max memory: 3.81 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
{"timestamp":"2026-09-08T23:15:54.920030Z","level":"ERROR","fields":{"message":"Error logging to file \"/home/wytcor/.cache/huggingface/xet/logs/xet_20260908T201554919-0300_105823.log\" (Permission denied (os error 13)); falling back to console logging."},"filename":"/home/runner/work/xet-core/xet-core/xet_runtime/src/logging/init.rs","line_number":64}
{"timestamp":"2026-09-08T23:15:57.764760Z","level":"ERROR","fields":{"message":"Error logging to file \"/h

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.9.3 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


Carrega o dataset local

In [6]:
from datasets import load_dataset

anon_dataset = load_dataset("json", data_files="/home/wytcor/PROJECTs/anon-lib/data/results/dados.jsonl", split="train")

anon_dataset_split = anon_dataset.train_test_split(test_size=0.3, seed=42)


train_dataset = anon_dataset_split["train"]
test_dataset = anon_dataset_split["test"]

# Exibe o número de linhas de cada um para conferir
print(f"Treino: {len(train_dataset)} linhas")
print(f"Teste: {len(test_dataset)} linhas")

Treino: 721 linhas
Teste: 309 linhas


In [7]:
train_dataset[0]

{'text': 'Em 07/12/1915, o sistema de segurança da Phillips, Harris and Torres alertou que o cartão 502097165273 foi usado pelo IP 204.167.114.55 em Levineton, enquanto Leslie Roberts enviou um documento com ID Y76670474 para bli@example.org e confirmou contato via 225.925.9578 no endereço USCGC Reed FPO AP 83293.',
 'entities': {'PERSON_NAME': 'Leslie Roberts',
  'EMAIL': 'bli@example.org',
  'ADDRESS': 'USCGC Reed FPO AP 83293',
  'DATE': '07/12/1915',
  'LOCATION': 'Levineton',
  'CREDIT_CARD': '502097165273',
  'PHONE': '225.925.9578',
  'COMPANY': 'Phillips, Harris and Torres',
  'IP_ADDRESS': '204.167.114.55',
  'DOCUMENT_ID': 'Y76670474'}}

Criando o formato da conversa com o dataset 

In [10]:
import json

def generate_conversation(examples):
    prompt = """
        Você é um detector de dados pessoais (PII). Devolva as entidades "
        "encontradas no texto em JSON, seguindo o schema pedido.\n"
        "Regras:\n"
        "- Copie cada valor exatamente como aparece no texto, sem reescrever.\n"
        "- Não devolva o texto mascarado: quem mascara é o código.\n"
        "- Se não houver nenhuma entidade, devolva uma lista vazia.\n"
        "- Capture TODOS os nomes de pessoas, incluindo terceiros mencionados "
        "por vínculo familiar, profissional ou social \n"
        "A lista de possíveis place holders: 
            "[PERSON_NAME]",
            "[EMAIL]",
            "[ADDRESS]",
            "[DATE]",
            "[LOCATION]",
            "[CREDIT_CARD]",
            "[DOCUMENT_ID]",
            "[PHONE]",
            "[COMPANY]",
            "[IP_ADDRESS]",
            
        Exemplos de entidades identificadas:
        "(ex: 'pai de', 'filha de', 'cônjuge de', 'advogado de').\n"
        "- Capture idades expressas como número seguido de 'anos' "
        "(ex: '35 anos', 'com 12 anos') como entidade do tipo AGE."
    """
    inputs_sentence_text = examples["text"]
    outputs_text = examples["entities"]
    conversas = []

    for input_text, output_text in zip(inputs_sentence_text, outputs_text):
        conversas.append([
            {"role" : "user", "content" : prompt+"\n\n"+input_text},
            {"role": "assistant", "content": json.dumps(output_text, ensure_ascii=False)},
        ])
    return { "conversations": conversas, }

In [20]:
from datasets import Dataset

train_anon_conversations = Dataset.from_dict({
    "text": tokenizer.apply_chat_template(
        list(train_dataset.map(generate_conversation, batched = True)["conversations"]),
        tokenize = False,
    )
})

In [22]:
test_anon_conversations = Dataset.from_dict({
    "text": tokenizer.apply_chat_template(
        list(test_dataset.map(generate_conversation, batched = True)["conversations"]),
        tokenize = False,
    )
})

In [23]:
test_anon_conversations[0]

{'text': '<|im_start|>user\n\n        Você é um detector de dados pessoais (PII). Devolva as entidades "\n        "encontradas no texto em JSON, seguindo o schema pedido.\n"\n        "Regras:\n"\n        "- Copie cada valor exatamente como aparece no texto, sem reescrever.\n"\n        "- Não devolva o texto mascarado: quem mascara é o código.\n"\n        "- Se não houver nenhuma entidade, devolva uma lista vazia.\n"\n        "- Capture TODOS os nomes de pessoas, incluindo terceiros mencionados "\n        "por vínculo familiar, profissional ou social \n"\n        "A lista de possíveis place holders: \n            "[PERSON_NAME]",\n            "[EMAIL]",\n            "[ADDRESS]",\n            "[DATE]",\n            "[LOCATION]",\n            "[CREDIT_CARD]",\n            "[DOCUMENT_ID]",\n            "[PHONE]",\n            "[COMPANY]",\n            "[IP_ADDRESS]",\n\n        Exemplos de entidades identificadas:\n        "(ex: \'pai de\', \'filha de\', \'cônjuge de\', \'advogado de\').\n

Configurações de Fine-tune

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_anon_conversations,
    eval_dataset = test_anon_conversations,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 3, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
        padding_free  = False, # Set to True if > 17 GB VRAM
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2): 100%|██████████| 309/309 [00:00<00:00, 357.91 examples/s]


In [34]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 721 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,587,520 of 600,637,440 (0.76% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.554300
2,0.509000
3,0.510000
4,0.548500
5,0.514500
6,0.495700
7,0.520900
8,0.534600
9,0.510900
10,0.558700


Inferência

In [42]:
prompt = """
        Você é um detector de dados pessoais (PII). Devolva as entidades "
        "encontradas no texto em JSON, seguindo o schema pedido.\n"
        "Regras:\n"
        "- Copie cada valor exatamente como aparece no texto, sem reescrever.\n"
        "- Não devolva o texto mascarado: quem mascara é o código.\n"
        "- Se não houver nenhuma entidade, devolva uma lista vazia.\n"
        "- Capture TODOS os nomes de pessoas, incluindo terceiros mencionados "
        "por vínculo familiar, profissional ou social \n"
        "A lista de possíveis place holders: 
            "[PERSON_NAME]",
            "[EMAIL]",
            "[ADDRESS]",
            "[DATE]",
            "[LOCATION]",
            "[CREDIT_CARD]",
            "[DOCUMENT_ID]",
            "[PHONE]",
            "[COMPANY]",
            "[IP_ADDRESS]",
            
        Exemplos de entidades identificadas:
        "(ex: 'pai de', 'filha de', 'cônjuge de', 'advogado de').\n"
        "- Capture idades expressas como número seguido de 'anos' "
        "(ex: '35 anos', 'com 12 anos') como entidade do tipo AGE."
    """

input_text="Em 23/11/1931, o Wyctor Fogos da Rocha enviou um e-mail para oda-rosa@example.com relatando que a reunião será realizada em Leão do Sul, no endereço Viaduto de Aragão, 3 Luxemburgo 73427744 Souza / RJ."

messages = [
    {"role" : "user", "content" : prompt+"\n\n"+input_text}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = True, # Habilita o bloco <think> (usado no treino, mesmo que vazio)
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1024, # Increase for longer outputs!
    temperature = 0.1, top_p = 0.9, top_k = 10, # For thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<think>

</think>

{"PERSON_NAME": "Wyctor Fogos da Rocha", "EMAIL": "oda-rosa@example.com", "ADDRESS": "Viaduto de Aragão, 3 Luxemburgo 73427744 Souiza / RJ", "DATE": "23/11/1931", "LOCATION": "Leão do Sul", "CREDIT_CARD": null, "PHONE": null, "COMPANY": null, "IP_ADDRESS": null, "DOCUMENT_ID": null}<|im_end|>


Salva o modelo

In [45]:
model.save_pretrained("../data/results/anon_qwen_lora")
tokenizer.save_pretrained("../data/results/anon_qwen_lora")

('../data/results/anon_qwen_lora/tokenizer_config.json',
 '../data/results/anon_qwen_lora/special_tokens_map.json',
 '../data/results/anon_qwen_lora/chat_template.jinja',
 '../data/results/anon_qwen_lora/vocab.json',
 '../data/results/anon_qwen_lora/merges.txt',
 '../data/results/anon_qwen_lora/added_tokens.json',
 '../data/results/anon_qwen_lora/tokenizer.json')

In [ ]:
# Merge to 16bit
if True:
    model.save_pretrained_merged("../data/results/anon_qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)

# Merge to 4bit
if True:
    model.save_pretrained_merged("../data/results/anon_qwen_finetune_4bit", tokenizer, save_method = "merged_4bit_forced",)


Unsloth: Downloading `unsloth/qwen3-0.6b` into the Hugging Face cache so future exports skip the 1.1GB download...
{"timestamp":"2026-09-09T00:01:29.918436Z","level":"ERROR","fields":{"message":"Error logging to file \"/home/wytcor/.cache/huggingface/xet/logs/xet_20260908T210129916-0300_105492.log\" (Permission denied (os error 13)); falling back to console logging."},"filename":"/home/runner/work/xet-core/xet-core/xet_runtime/src/logging/init.rs","line_number":64}
Found HuggingFace hub cache directory: /home/wytcor/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `../data/results/anon_qwen_finetune_16bit`: 100%|██████████| 1/1 [00:00<00:00,  1.61it/s]


Successfully copied all 1 files from cache to `../data/results/anon_qwen_finetune_16bit`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:13<00:00, 13.43s/it]


Unsloth: Merge process complete. Saved to `/home/wytcor/PROJECTs/anon-lib/data/results/anon_qwen_finetune_16bit`


RuntimeError: Unsloth: Merging into 4bit will cause your model to lose accuracy if you plan
to merge to GGUF or others later on. I suggest you to do this as a final step
if you're planning to do multiple saves.
If you are certain, change `save_method` to `merged_4bit_forced`.